## Metareader – experimental resolution for X-Ray and Cryo-EM structures

The goal of this notebook is to show how the `metareader` script can be used to:

- identify the experimental method used to solve a structure,
- check the experimental resolution from mmCIF metadata.

### Two example structures are used:

- **1UBQ** – structure solved by X-Ray diffraction,
- **7K00** – structure solved by Cryo-EM.

Input files are local mmCIF files stored in: `data/pdb/`


## Example 1: X-Ray structure (1UBQ)


### Listing available metadata categories

Before extracting any information, we check which metadata categories are present
in an mmCIF file.


In [28]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/1ubq.cif \
  --list-categories | head -n 30

entry
audit_conform
database_2
pdbx_audit_revision_history
pdbx_audit_revision_details
pdbx_audit_revision_group
pdbx_audit_revision_category
pdbx_audit_revision_item
pdbx_database_status
audit_author
citation
citation_author
entity
entity_poly
pdbx_entity_nonpoly
entity_poly_seq
entity_src_gen
chem_comp
pdbx_poly_seq_scheme
pdbx_nonpoly_scheme
software
cell
symmetry
exptl
exptl_crystal
diffrn
diffrn_radiation
diffrn_radiation_wavelength
refine
refine_hist


From the list of available categories we can see that this structure was solved by X-ray
diffraction. This is indicated by the presence of categories such as:

- `exptl`
- `diffrn`
- `refine`

and by the absence of any EM-related categories. Therefore, the experimental resolution
should be read from the `refine` category.


### Checking the experimental method

The experimental method is stored in the `exptl` category.


In [29]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/1ubq.cif \
  -c exptl \
  > ./outputs/1ubq_exptl.json

In [30]:
import json

with open("./outputs/1ubq_exptl.json") as f:
    data = json.load(f)

data["exptl"][0]["method"]

'X-RAY DIFFRACTION'

In the output, the experimental method is stored in the `exptl` category.
We look for the field `method`:

"exptl":[{"entry_id":"1UBQ",**"method"**:"X-RAY DIFFRACTION"

This confirms that the structure 1UBQ was solved using **X-ray diffraction**.


### Checking the experimental resolution

For X-ray structures the experimental resolution is stored in the `refine` category,
most commonly in the field `ls_d_res_high`.


In [31]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/1ubq.cif \
  -c refine \
  > ./outputs/1ubq_refine.json

In [32]:
with open("./outputs/1ubq_refine.json") as f:
    data = json.load(f)

data["refine"][0]["ls_d_res_high"]

'1.8'

In the output, the experimental resolution is stored in the `refine` category.
We look for the field `ls_d_res_high`:


"refine": [
{
"entry_id": "1UBQ",
"ls_number_reflns_obs": "?",
"ls_number_reflns_all": "?",
"pdbx_ls_sigma_I": "?",
"pdbx_ls_sigma_F": "?",
"pdbx_data_cutoff_high_absF": "?",
"pdbx_data_cutoff_low_absF": "?",
"pdbx_data_cutoff_high_rms_absF": "?",
"ls_d_res_low": "?",
"ls_d_res_high": "1.8"
}
]

The experimental resolution for the X-ray structure is given by the field
`ls_d_res_high`, which in this case equals **1.8 Å**.


### For 1UBQ:

- method: read from `exptl -> method`,
- resolution: read from `refine -> ls_d_res_high`.


## Example 2: Cryo-EM structure (7K00)


### Listing available categories


In [33]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/7k00.cif \
  --list-categories | head -n 30

entry
audit_conform
database_2
pdbx_audit_revision_history
pdbx_audit_revision_details
pdbx_audit_revision_group
pdbx_audit_revision_category
pdbx_audit_revision_item
pdbx_database_status
pdbx_database_related
audit_author
citation
citation_author
entity
entity_name_com
entity_poly
pdbx_entity_nonpoly
entity_poly_seq
entity_src_nat
pdbx_entity_src_syn
chem_comp
pdbx_poly_seq_scheme
pdbx_nonpoly_scheme
pdbx_unobs_or_zero_occ_atoms
cell
symmetry
exptl
struct
struct_keywords
struct_asym


In Cryo-EM structures the set of categories is different from X-ray entries.
The `refine` and `diffrn` categories are usually absent, while categories related
to electron microscopy and 3D reconstruction are present.


### Checking the experimental method


In [34]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/7k00.cif \
  -c exptl \
  > ./outputs/7k00_exptl.json

In [35]:
with open("./outputs/7k00_exptl.json") as f:
    data = json.load(f)

data["exptl"][0]["method"]

'ELECTRON MICROSCOPY'

In the output we again look for:

exptl -> method


"exptl": [
{
"absorpt_coefficient_mu": "?",
"absorpt_correction_T_max": "?",
"absorpt_correction_T_min": "?",
"absorpt_correction_type": "?",
"absorpt_process_details": "?",
"entry_id": "7K00",
"crystals_number": "?",
"details": "?",
"method": "ELECTRON MICROSCOPY"
}
]


This confirms that the structure 7K00 was solved using **Cryo-EM**.


### Checking the experimental resolution


In [36]:
!python ../src/rnapolis/metareader.py \
  ./data/pdb/7k00.cif \
  -c em_3d_reconstruction \
  > ./outputs/7k00_resolution.json

In [37]:
with open("./outputs/7k00_resolution.json") as f:
    data = json.load(f)

data["em_3d_reconstruction"][0]["resolution"]

'1.98'

In Cryo-EM entries the experimental resolution is typically stored in a `em_3d_reconstruction` category. We look for the `resolution` field:


"em_3d_reconstruction": [
{
"entry_id": "7K00",
"id": "1",
"algorithm": "?",
"details": "Ewald sphere corrected in RELION",
"refinement_type": "?",
"image_processing_id": "1",
"num_class_averages": "?",
"num_particles": "307495",
"resolution": "1.98"
}
]


In this case the experimental resolution of the structure equals **1.98 Å**


### For 7K00:

- method: read from `exptl -> method`,
- resolution: read from `em_3d_reconstruction -> resolution`.


## Summary

Using `metareader`, experimental information can be extracted directly from
mmCIF metadata:

For X-Ray structures (e.g. 1UBQ):

- method: read from `exptl -> method`,
- resolution: read from `refine -> ls_d_res_high`.

For Cryo-EM structures (e.g. 7K00):

- method: read from `exptl -> method`,
- resolution: read from `em_3d_reconstruction -> resolution`.


This shows how experimental resolution can be verified for both X-Ray and Cryo-EM
structures using the same tool, based only on the metadata stored in the mmCIF files.
